# MiniCPM-V 手机巡检单图基线

先用授权图片验证单图理解和结构输出，再在手机验收。此 Notebook 使用公开的 4.6 路线，尚未执行真实图片推理。完整业务流见 Part 5。

In [ ]:
from pathlib import Path
import sys, subprocess, json
candidates=[Path.cwd(), Path.cwd().parent, Path.cwd()/"minicpm-purplebook-draft"]
ROOT=next((p.resolve() for p in candidates if (p/"examples/task_contract.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("请先上传并解压完整实践包，再在项目目录打开Notebook")
sys.path.insert(0,str(ROOT/"examples"))
def run(args):
    subprocess.run([str(x) for x in args],cwd=ROOT,check=True)
print("Project:",ROOT)


## 1 准备推理依赖
保留镜像中匹配的 PyTorch 与 torchvision。安装后重启内核。

In [ ]:
run([sys.executable,"-m","pip","install","transformers==5.7.0","modelscope==1.33.0",
     "accelerate","Pillow","torchvision","av"])


## 2 从魔搭下载固定模型
使用完整40位Git提交。ModelScope 1.33.0的snapshot_download不接受该仓库的提交哈希；配套下载器使用固定文件接口并逐文件校验SHA256。


In [ ]:
from download_vision_model import download_fixed
revision=input("MiniCPM-V 4.6完整Git提交（回车使用已验证提交）：").strip() or "fd02313ad21ac485c595cb01231a3bded449cff5"
model_dir=download_fixed(revision, ROOT/"models/minicpm-v-4.6")
(ROOT/"results").mkdir(exist_ok=True)
(ROOT/"results/vision-config.json").write_text(json.dumps(
    {"model":"OpenBMB/MiniCPM-V-4.6","requested_revision":revision,
     "downsample_mode":"16x","max_slice_nums":1,"enable_thinking":False},indent=2),encoding="utf-8")


## 3 上传自有或授权照片
使用真实存在的本地路径。图片未包含在教学包中。

In [ ]:
image_path=Path(input("授权图片的本地路径：").strip()).expanduser().resolve()
assert image_path.is_file()
import hashlib
with image_path.open("rb") as f: image_hash=hashlib.file_digest(f,"sha256").hexdigest()
print("Image SHA256:",image_hash)
run([sys.executable,"examples/vision_baseline.py","--model",model_dir,
     "--image",image_path,"--downsample","16x","--out","results/inspection-one.json"])


## 4 审阅原始输出
结构合法不代表视觉识别正确。人工对照照片判断对象、现象与无依据补充。

In [ ]:
result=json.loads((ROOT/"results/inspection-one.json").read_text(encoding="utf-8"))
print(json.dumps(result,ensure_ascii=False,indent=2))


## 5 用同一图片比较 4x
这一调用必须同时改变预处理与模型生成的 downsample_mode。

In [ ]:
run([sys.executable,"examples/vision_baseline.py","--model",model_dir,
     "--image",image_path,"--downsample","4x","--out","results/inspection-one-4x.json"])


## 6 继续手机与业务验收

使用官方 Apps 固定版本和相匹配的 LLM / mmproj，接入 `android/InspectionRunner.kt` 与本地记录组件。按 Part 5 的验收矩阵测试飞行模式、新照片、人工确认、离线保存和显式同步。

电脑模拟台账可先运行 `examples/ledger.py`；它不承担图像推理。JSON 教学夹具只用于验证接口，不是视觉测试结果。